# Photoelectric effect and light quanta

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/photoelectric_effect)

Einstein's 1905 light-quantum paper: light of frequency $\nu$ arrives in packets of energy $h\nu$, and one packet frees one electron with kinetic energy $K = h\nu - \phi - \varepsilon$. A batch of photons × frequencies × intensities shows the threshold $\nu_0 = \phi/h$, a stopping potential $eV_s = h\nu - \phi$ that ignores the intensity, and no time delay — all against the classical wave picture. A straight-line fit to Millikan's 1916 sodium measurements gives back Planck's constant.

**Vector form**

$$K = h\nu - \phi - \varepsilon,\qquad eV_s = h\nu - \phi,\qquad \nu_0 = \phi/h,\qquad I(V) = \Phi\,P(K > eV)$$

**Index (tensor) form**

$$K_{ifn} = h\,\nu_f - \phi - \varepsilon_{ifn},\qquad V_{s,if} = \max_n K_{ifn}/e,\qquad \min_{a,b}\sum_m \big(V_m - a\,\nu_m - b\big)^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\nu_f$ | `frequencies(3.5, 13.5, 91)` | (F,) | light frequencies in $10^{14}$ Hz |
| $\varepsilon_{ifn}$ | `electron_depths(F, N, n_int=I)` | (I, F, N) | depth of the electron that absorbs photon $n$, uniform on $[0, 3]$ eV ($i$: intensity) |
| $h\nu_f - \phi - \varepsilon_{ifn}$ | `photon_energy(nu)[:, None] - phi - depth` | (I, F, N) | one broadcast: every photon of every frequency and intensity at once; $K \le 0$ ⇒ no emission |
| $P(K > eV)$ | `n_max - searchsorted(K_sorted, V)` | (I, F, V) | electrons that climb the retarding voltage $V$: the photocurrent |
| $V_{s,if}$ | `K.max(dim=-1).values` | (I, F) | stopping potential: the fastest electron; equal for every intensity $i$ |
| $a, b$ | `torch.linalg.lstsq(A, y)` | (2,) | straight line through Millikan's points: slope $a = h/e$, intercept $b = -\phi/e$ |
| $t_{\text{wave}} = \phi/(S A)$ | `classical_delay(S, phi, area)` | () | wave picture: time to soak up $\phi$ over one atom's area $A$ from irradiance $S$ |

This is one of Einstein's four 1905 "miracle year" papers: light quanta and the photoelectric effect (this problem; Nobel Prize 1921, "especially for his discovery of the law of the photoelectric effect"), Brownian motion (thermodynamics: Random walk and diffusion), and special relativity with $E = mc^2$ (astro: Einstein 1905: moving bodies and $E = mc^2$). The wave picture fails three ways: it gives emission at any frequency, kinetic energies that grow with the intensity, and minutes of delay for dim light (an atom collecting $\phi$ from $S = 10$ mW/m²) — while experiment shows a sharp threshold, $V_s$ independent of intensity and no measurable delay (below 3 ns, Lawrence and Beams 1928). Brighter light only means more photons, so the current grows but $V_s$ does not move. The red squares are Millikan's 1916 stopping potentials for a sodium surface cut in vacuum (R. A. Millikan, Phys. Rev. 7, 355 (1916), four mercury lines, transcribed in data/millikan_1916_sodium.csv from the paper as tabulated in a Roskilde University problem set; a 1916 publication, public domain). Millikan set out to refute Einstein's equation and confirmed it; his slope gives $h$ within about 0.3 % of today's exact value $6.62607015\times10^{-34}$ J s. The intercept, about 1.8 eV, is not the clean-sodium work function (2.3 eV): it contains the contact potential of the apparatus, and the surface was probably contaminated (D. Lloyd, Am. J. Phys. 2015). The simulated metal is idealised (a flat band of depths, no angles, no temperature): the straight line $eV_s = h\nu - \phi$ is exact, the shape of the current curves is only schematic.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/photoelectric_effect/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    phi, S, n_photons = params["phi"], params["irradiance"], params["n_photons"]
    torch.manual_seed(0)

    # Monte Carlo: every frequency of the sweep × 3 intensities × every photon, in one batch
    nu = tensors.frequencies(3.5, 13.5, 91)  # 10^14 Hz
    intensities = torch.tensor([0.5, 1.0, 2.0])
    voltages = tensors.retarding_voltages(-1.0, 4.0)
    sim = solve.simulate(nu, phi, intensities, n_photons, voltages=voltages)

    synthetic = solve.fit_line(nu, sim["V_stop"][1])
    nu0 = equation.threshold_frequency(phi)
    print(f"threshold ν0 = φ/h = {nu0:.3f}×10¹⁴ Hz  (λ0 = {2997.92458 / nu0:.0f} nm)")
    print(
        f"simulated stopping potentials: slope {synthetic['slope']:.5f} V/10¹⁴ Hz (h/e = {equation.H_OVER_E:.5f}),"
        f" intercept {synthetic['intercept']:.3f} V (−φ/e = {-phi:.3f})"
    )
    spread = (sim["V_stop"] - sim["V_stop"][1]).abs().max()
    print(
        f"stopping potential at 0.5×, 1×, 2× intensity differs by at most {spread:.4f} V; saturation currents"
        f" {sim['current'][:, -1, 0].tolist()}"
    )

    # Millikan (1916), sodium
    nu_m, V_m = tensors.millikan_sodium()
    fit = solve.fit_line(nu_m, V_m)
    h = fit["slope"] * 1e-14 * equation.E_CHARGE
    print(f"Millikan 1916 sodium, {len(nu_m)} lines: slope {fit['slope']:.4f} ± {fit['slope_err']:.4f} V/10¹⁴ Hz")
    print(f"  h = {h:.4e} J s  vs today {equation.H_PLANCK:.5e} J s  ({100 * (h / equation.H_PLANCK - 1):+.2f} %)")
    nu0_m = -fit["intercept"] / fit["slope"]
    print(f"  intercept: φ = {-fit['intercept']:.2f} ± {fit['intercept_err']:.2f} eV, ν0 = {nu0_m:.2f}×10¹⁴ Hz")

    area = math.pi * (1e-10) ** 2  # one atom
    t_wave = equation.classical_delay(S, phi, area)
    print(
        f"wave picture at {S:g} W/m²: an electron needs {t_wave:.0f} s ({t_wave / 60:.1f} min) to collect φ;"
        " observed delay < 3 ns (Lawrence & Beams 1928)"
    )
    return plot.animate(nu, sim, phi, intensities, (nu_m, V_m, fit), t_wave, out_dir / "photoelectric_effect.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/photoelectric_effect/equation.py
r"""Physics → tensor translation.

Einstein (1905): light of frequency ν arrives in quanta of energy hν. One quantum is absorbed by one electron, which
spends the work function φ (plus its depth ε ≥ 0 below the top of the occupied band) to leave the metal:

    K_{fn} = h\nu_f - \phi - \varepsilon_{fn}          (f: frequency, n: electron — a broadcast, no loop)

The fastest electrons (ε = 0) are stopped by a retarding potential V_s with

    e V_s = h\nu - \phi ,

a straight line in ν with slope h/e and threshold ν_0 = φ/h, whatever the intensity. With ε uniform on [0, w] the
current that still reaches the collector at retarding voltage V is

    I(V) = \Phi\,\operatorname{clip}\big((h\nu - \phi - eV)/w,\ 0,\ 1\big)      (V ≥ 0; Φ: photon flux × yield)

Units: energies in eV, potentials in V, frequencies in 10^14 Hz, so h/e = 0.41357 V per 10^14 Hz.

The classical wave picture instead gives the electron energy at a rate S·A (irradiance × atomic area), so it must
wait t = φ/(S A) before it can leave, and the kinetic energy grows with the intensity, not with ν.
"""

import torch

H_PLANCK = 6.62607015e-34  # J s (exact, SI 2019)
E_CHARGE = 1.602176634e-19  # C (exact)
H_OVER_E = H_PLANCK / E_CHARGE * 1e14  # V per 10^14 Hz ≈ 0.41357


def photon_energy(nu: torch.Tensor) -> torch.Tensor:
    """hν in eV for ν in 10^14 Hz: any shape."""
    return H_OVER_E * nu


def threshold_frequency(phi: float) -> float:
    """ν0 = φ/h in 10^14 Hz."""
    return phi / H_OVER_E


def kinetic_energy(nu: torch.Tensor, phi: float, depth: torch.Tensor) -> torch.Tensor:
    r"""K_{fn} = hν_f − φ − ε_{fn} (eV).   nu: (F,), depth: (..., F, N) → (..., F, N); K ≤ 0: no emission."""
    return photon_energy(nu)[:, None] - phi - depth


def stopping_potential(nu: torch.Tensor, phi: float) -> torch.Tensor:
    """V_s = (hν − φ)/e, zero below threshold: (F,)."""
    return torch.clamp(photon_energy(nu) - phi, min=0.0)


def photocurrent(V: torch.Tensor, nu: torch.Tensor, phi: float, flux: float, width: float) -> torch.Tensor:
    """Expected current at retarding voltage V (broadcast V against nu), depth ε uniform on [0, width].

    An accelerating voltage (V < 0) collects every emitted electron but cannot create any below threshold."""
    return flux * torch.clamp((photon_energy(nu) - phi - torch.clamp(V, min=0.0)) / width, 0.0, 1.0)


def classical_delay(S: float, phi: float, area: float) -> float:
    """Wave picture: seconds an electron needs to soak up φ (eV) from irradiance S (W/m²) over an area (m²)."""
    return phi * E_CHARGE / (S * area)


def classical_kinetic_energy(intensity: torch.Tensor, K1: float) -> torch.Tensor:
    """Wave picture: the electron is shaken by the field E0, so K ∝ E0² ∝ intensity (scaled so K(1) = K1)."""
    return K1 * intensity


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/photoelectric_effect/tensors.py
"""Build the concrete tensors: frequencies, random electron depths, photon counts and Millikan's 1916 data."""

import csv
from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "millikan_1916_sodium.csv"
C_LIGHT = 299792458.0  # m/s
BAND = 3.0  # eV: depth range of the occupied electron states below the top (sodium's band is about 3 eV wide)


def frequencies(nu_min: float, nu_max: float, n: int) -> torch.Tensor:
    """Light frequencies (F,) in units of 10^14 Hz."""
    return torch.linspace(nu_min, nu_max, n, dtype=torch.float64)


def electron_depths(n_freq: int, n_photons: int, width: float = BAND, seed: int = 0, n_int: int | None = None):
    """Depth ε ≥ 0 of the electron that absorbs each photon, uniform on [0, width]: (F, N), or (I, F, N) with n_int."""
    g = torch.Generator().manual_seed(seed)
    shape = (n_freq, n_photons) if n_int is None else (n_int, n_freq, n_photons)
    return width * torch.rand(shape, generator=g, dtype=torch.float64)


def photon_counts(intensities: torch.Tensor, n_photons: int, seed: int = 0) -> torch.Tensor:
    """Photons absorbed during one exposure at each relative intensity: Poisson(n_photons · intensity), shape (I,)."""
    g = torch.Generator().manual_seed(seed + 1)
    return torch.poisson(n_photons * intensities.double(), generator=g).long()


def retarding_voltages(v_min: float, v_max: float, n: int = 121) -> torch.Tensor:
    """Collector voltages (V,) in volts: negative accelerates, positive retards the electrons."""
    return torch.linspace(v_min, v_max, n, dtype=torch.float64)


def millikan_sodium() -> tuple[torch.Tensor, torch.Tensor]:
    """Millikan (1916), sodium: frequencies ν = c/λ (M,) in 10^14 Hz and stopping potentials (M,) in volts."""
    with DATA.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(line for line in f if not line.startswith("#")))
    lam = torch.tensor([float(r["wavelength_nm"]) for r in rows], dtype=torch.float64) * 1e-9
    V = torch.tensor([float(r["stopping_potential_V"]) for r in rows], dtype=torch.float64)
    return C_LIGHT / lam / 1e14, V


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/photoelectric_effect/solve.py
"""Monte Carlo of photons ejecting electrons, retarding-voltage curves, and the straight-line fit eV_s = hν − φ."""

import torch

from . import equation, tensors


def measured_stopping_potential(K: torch.Tensor) -> torch.Tensor:
    """The retarding voltage that stops even the fastest emitted electron: max K over the last axis (≥ 0)."""
    return torch.clamp(K.max(dim=-1).values, min=0.0)


def simulate(nu, phi, intensities, n_photons, width=tensors.BAND, seed=0, voltages=None) -> dict:
    """Every (intensity, frequency, photon) at once.

    Returns kinetic energies K (I, F, N) (−inf for photons not absorbed in this exposure or not emitted), the
    stopping potentials (I, F) and the current (I, F, V) reaching the collector, in units of n_photons.
    """
    n_int = len(intensities)
    counts = tensors.photon_counts(intensities, n_photons, seed)  # (I,)
    n_max = int(counts.max())
    depth = tensors.electron_depths(len(nu), n_max, width, seed, n_int=n_int)  # (I, F, N)
    K = equation.kinetic_energy(nu, phi, depth)  # (I, F, N): hν_f − φ − ε
    present = torch.arange(n_max)[None, None, :] < counts[:, None, None]  # (I, 1, N) photons of this exposure
    K = torch.where(present & (K > 0), K, torch.tensor(-torch.inf, dtype=K.dtype))
    if voltages is None:
        voltages = tensors.retarding_voltages(-1.0, float(equation.photon_energy(nu).max()) - phi + 0.5)
    K_sorted = K.sort(dim=-1).values
    V = voltages.expand(n_int, len(nu), -1).contiguous()  # (I, F, V)
    # electrons with K > eV pass: N − (number with K ≤ eV)
    passed = n_max - torch.searchsorted(K_sorted, V, right=True)
    return {
        "K": K,
        "V_stop": measured_stopping_potential(K),
        "current": passed.double() / n_photons,
        "voltages": voltages,
        "counts": counts,
    }


def fit_line(nu: torch.Tensor, V: torch.Tensor) -> dict:
    """Least squares V = a ν + b (only points above threshold, V > 0), with standard errors from the residuals."""
    keep = V > 0
    x, y = nu[keep], V[keep]
    A = torch.stack([x, torch.ones_like(x)], dim=-1)  # (M, 2) design matrix
    coef = torch.linalg.lstsq(A, y[:, None]).solution[:, 0]  # (2,)
    resid = y - A @ coef
    dof = max(len(x) - 2, 1)
    cov = torch.linalg.inv(A.T @ A) * (resid @ resid) / dof  # (2, 2)
    return {
        "slope": float(coef[0]),
        "intercept": float(coef[1]),
        "slope_err": float(cov[0, 0].sqrt()),
        "intercept_err": float(cov[1, 1].sqrt()),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/photoelectric_effect/plot.py
"""Sweep the light frequency: electron energies, retarding-voltage curves and the line eV_s = hν − φ."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation

AMBER, GREY = "#f59e0b", "#9ca3af"
INT_COLORS = [ACCENT[5], ACCENT[0], ACCENT[3]]


def animate(nu, sim, phi, intensities, millikan, t_wave, path: Path) -> Path:
    nu_np = nu.numpy()
    K = sim["K"].numpy()  # (I, F, N), −inf where no electron
    current = sim["current"].numpy()  # (I, F, V)
    volts = sim["voltages"].numpy()
    V_stop = sim["V_stop"].numpy()  # (I, F)
    nu_m, V_m, fit = (t.numpy() if hasattr(t, "numpy") else t for t in millikan)
    exact = equation.stopping_potential(nu, phi).numpy()
    k_top = max(float(exact.max()), 0.5)

    fig, (ax_k, ax_iv, ax_line) = plt.subplots(1, 3, figsize=(10.5, 4))

    # --- electron kinetic energies at intensity 1
    bins = np.linspace(0, k_top + 0.2, 31)
    ax_k.set_xlim(bins[0], bins[-1])
    ax_k.set_xlabel("kinetic energy $K$ (eV)")
    ax_k.set_ylabel("electrons per bin")
    kmax_line = ax_k.axvline(0, color=AMBER, lw=2, label="$h\\nu - \\phi$")
    ax_k.legend(fontsize=8, loc="upper right")
    none_text = ax_k.text(0.5, 0.5, "", transform=ax_k.transAxes, ha="center", fontsize=10, color=ACCENT[1])
    peak = max(1, int(np.isfinite(K[1]).sum(axis=-1).max() / 30 * 1.6))
    ax_k.set_ylim(0, peak)
    bars = ax_k.bar(bins[:-1], np.zeros(len(bins) - 1), width=np.diff(bins), align="edge", color=ACCENT[0], alpha=0.75)

    # --- retarding-voltage curves for 3 intensities
    ax_iv.set_xlim(volts[0], volts[-1])
    ax_iv.set_ylim(0, current.max() * 1.08 + 1e-9)
    ax_iv.set_xlabel("retarding voltage $V$ (V)")
    ax_iv.set_ylabel("photocurrent (photons → electrons)")
    iv_lines = [
        ax_iv.step([], [], where="post", color=c, lw=1.8, label=f"intensity × {float(s):g}")[0]
        for c, s in zip(INT_COLORS, intensities)
    ]
    vs_line = ax_iv.axvline(0, color=AMBER, ls=(0, (4, 3)), lw=1.5, label="$V_s$ (same for all)")
    ax_iv.legend(fontsize=7, loc="upper right")

    # --- stopping potential against frequency
    ax_line.set_xlim(nu_np[0], nu_np[-1])
    ax_line.set_ylim(-0.1, max(exact.max(), V_m.max()) + 0.6)
    ax_line.set_xlabel("frequency $\\nu$ ($10^{14}$ Hz)")
    ax_line.set_ylabel("stopping potential $V_s$ (V)")
    ax_line.plot(nu_np, exact, color=GREY, lw=1, ls=(0, (4, 3)), label=f"$hν-φ$, φ = {phi:.2f} eV")
    for c, s in zip(INT_COLORS, intensities):  # wave picture: V_s set by intensity, any ν
        ax_line.axhline(0.6 * float(s), color=c, lw=0.8, ls=":", alpha=0.8)
    ax_line.text(nu_np[-1] - 0.1, 1.25, "wave picture:\n$V_s \\propto$ intensity", ha="right", fontsize=7, color=GREY)
    xs = np.array([nu_np[0], nu_np[-1]])
    ax_line.plot(
        xs,
        fit["slope"] * xs + fit["intercept"],
        color=ACCENT[1],
        lw=1.2,
        label=f"fit: h = {fit['slope'] * 1e-14 * equation.E_CHARGE:.3e} J s",
    )
    ax_line.plot(nu_m, V_m, "s", color=ACCENT[1], ms=6, label="Millikan 1916, Na")
    sim_dots = [ax_line.plot([], [], "o", color=c, ms=ms)[0] for c, ms in zip(INT_COLORS, [6, 4, 2])]
    ax_line.legend(fontsize=7, loc="upper left")
    footer = fig.text(0.5, 0.01, "", ha="center", fontsize=9)

    def update(f: int):
        nu_f = nu_np[f]
        k = K[1, f][np.isfinite(K[1, f])]
        for bar, height in zip(bars, np.histogram(k, bins=bins)[0]):
            bar.set_height(height)
        kmax_line.set_xdata([exact[f]] * 2)
        none_text.set_text("" if len(k) else "below threshold:\nno electrons at any intensity")
        ax_k.set_title(f"ν = {nu_f:.2f}×10¹⁴ Hz: {len(k)} electrons", fontsize=9)
        for line, c in zip(iv_lines, current[:, f]):
            line.set_data(volts, c)
        vs_line.set_xdata([V_stop[1, f]] * 2)
        ax_iv.set_title(f"$V_s$ = {V_stop[1, f]:.2f} V   (exact {exact[f]:.2f} V)", fontsize=9)
        for dots, Vs in zip(sim_dots, V_stop):
            dots.set_data(nu_np[: f + 1], Vs[: f + 1])
        ax_line.set_title("$eV_s = h\\nu - \\phi$: slope $h/e$", fontsize=9)
        footer.set_text(
            f"λ = {2997.92458 / nu_f:.0f} nm,  hν = {equation.H_OVER_E * nu_f:.2f} eV,  φ = {phi:.2f} eV.   "
            f"Wave picture: wait {t_wave / 60:.0f} min to collect φ — observed: no delay (< 3 ns)"
        )

    update(0)  # titles in place before the layout is computed
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    frames = frame_indices(len(nu_np), 70) + [len(nu_np) - 1] * 10
    return save_gif(fig, update, frames, path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'phi': 2.28,
    'irradiance': 0.01,
    'n_photons': 3000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.photoelectric_effect")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.photoelectric_effect.main")
problem_dir = Path("problems/quantum/photoelectric_effect")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
